# deepEmulator — Coral training on Colab via `make`

Same outcome as notebook 08, different orchestration. This notebook drives the full pipeline through the project's Makefile — one `!make X` per stage. Pick the path you want and run the cells top-to-bottom.

**Two paths:**
- **Pixel baseline** — just `make train_pixel`. One command, ~30-60 min on T4, no DINO.
- **DINO pipeline** — `make collect_frames` → `make pretrain_dino` → `make train_encoder`. Three sequential commands, ~2-3 hours total on T4.

**Prereqs in Drive:**
- `MyDrive/deepEmulator/` — repo synced (see README's "Install on Colab without GitHub" section, or run `make sync_to_drive` locally).
- `MyDrive/deepEmulator/roms/PokemonCoral.gbc` — the ROM.
- `MyDrive/deepEmulator/states/coral_init.state` — recommended (see notebook 07 to record one). Without it the pipeline still runs, but the agent is stuck in boot phase and doesn't learn meaningfully.

In [ ]:
# Cell 2 — GPU check + system deps
!nvidia-smi -L || echo 'NO GPU — runtime > change runtime type > T4 GPU'
!pip install -q pyboy torch numpy

In [ ]:
# Cell 3 — Mount Drive, cd into the synced repo, install deepEmulator + dev/viz extras
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/deepEmulator
!make install_dev

In [ ]:
# Cell 4 — Show the workflow menu
!make help

## Sanity checks (run before any long training)

Cell 5 prints the Coral RAM state via `dump_state`. If you see nonsense values (`party_count: 42`, `badges: 255`) one or more `# VERIFY` RAM constants in `cartridges/pokemon_crystal.py` is wrong for Coral — patch and re-sync before training.

Cell 6 runs 200 random env steps and confirms phased reward fires. Expected: `sum=2.000`, `phase: boot` when no init.state is present, or `phase: tutorial` if init.state is loaded.

In [ ]:
# Cell 5 — verify RAM
!make verify_ram

In [ ]:
# Cell 6 — 200-step smoke
!make smoke_rom

## Path A — Pixel baseline

Skip this section if you're doing the DINO pipeline instead. This trains DDQN directly on raw pixels — the Phase I baseline. ~30-60 min on T4 for 100K steps.

If you want to A/B against DINO later, save the bundle path printed at the end (also recoverable via `make latest_bundle`).

In [ ]:
# Cell 7 — pixel-baseline DDQN. Re-running auto-resumes from latest.txt.
!make train_pixel STEPS=100000 SAVE_EVERY=10000

In [ ]:
# Cell 8 — save the pixel-baseline path for later eval (optional)
import os
PIXEL_RUN = open('checkpoints/pokemon_coral/latest.txt').read().strip()
os.environ['PIXEL_RUN'] = PIXEL_RUN
print('PIXEL_RUN =', PIXEL_RUN)
!make metrics

## Path B — DINO pipeline (three sequential trainings)

1. **Collect frames** — random-policy emulator frames into a corpus. ~5-10 min for 20K frames.
2. **Pretrain DINO** — fit ViT-tiny with self-supervised multi-crop loss. ~30-60 min on T4.
3. **Train encoder-DDQN** — DDQN with frozen DINO latents as input. ~30-60 min on T4.

For multi-cartridge DINO (trains once, reusable across Coral / Crystal / Red), use `make collect_multi` instead of `make collect_frames` in Cell 9.

In [ ]:
# Cell 9 — collect frames for DINO
!make collect_frames N_FRAMES=20000
# Or for cross-game DINO:
# !make collect_multi N_FRAMES=20000

In [ ]:
# Cell 10 — pretrain DINO ViT-tiny
!make pretrain_dino DINO_STEPS=20000 DINO_BATCH=64

In [ ]:
# Cell 11 — inspect DINO loss curve
!make latest_encoder
!make dino_metrics

In [ ]:
# Cell 12 — train DDQN on the frozen DINO latents
!make train_encoder STEPS=100000 SAVE_EVERY=10000

In [ ]:
# Cell 13 — save the DINO-treatment path for eval
import os
TREAT_RUN = open('checkpoints/pokemon_coral/latest.txt').read().strip()
os.environ['TREAT_RUN'] = TREAT_RUN
print('TREAT_RUN =', TREAT_RUN)
!make metrics

## Visualization + head-to-head eval

Once one or both training paths have produced bundles, render the visualizations and compare.

In [ ]:
# Cell 14 — trajectory arrows over the Crystal map (Coral uses same layout)
!make visualize

In [ ]:
# Cell 15 — attention rollout GIF (requires a trained DINO encoder)
!make attention

In [ ]:
# Cell 16 — head-to-head A/B (only if both bundles exist)
import os
if 'PIXEL_RUN' in os.environ and 'TREAT_RUN' in os.environ:
    !make eval PIXEL=$PIXEL_RUN TREAT=$TREAT_RUN
else:
    print('Run Path A and Path B first to populate PIXEL_RUN and TREAT_RUN.')

In [ ]:
# Cell 17 — render results inline
from IPython.display import Image, HTML, display
from pathlib import Path
if Path('arrows_pokemon_coral.png').exists():
    display(Image('arrows_pokemon_coral.png'))
if Path('attention_pokemon_coral.gif').exists():
    display(Image('attention_pokemon_coral.gif'))
if Path('eval_report.html').exists():
    display(HTML(open('eval_report.html').read()))

## Honest interpretation guide

**Success looks like**:
- `make metrics` shows `MeanLoss > 0` after step 1000 (= DDQN is training).
- With `coral_init.state` present: `MeanReward` grows beyond ~20/episode over time.
- `make latest_bundle` always points at a valid bundle dir.
- Bundles auto-save to Drive every `SAVE_EVERY` steps; re-running the training cell resumes from `latest.txt`.

**Failure modes**:
- `MeanLoss == 0` → burnin (1000 steps) not complete or `learn()` not firing.
- `MeanLoss == NaN` → LR too high; reduce in `DDQNConfig` or check that F9.7 uint8 buffer fix is in place.
- `MeanReward` flat at ~20 forever → expected without init.state (boot-phase only).
- `make verify_ram` printed nonsense → patch `# VERIFY` constants in `cartridges/pokemon_crystal.py`, re-sync Drive (`make sync_to_drive` locally), re-run.

**To reset and start over**: delete `checkpoints/pokemon_coral/` from Drive, re-run from Cell 7. DINO encoder is preserved (lives under `encoders/`).